# 03 - Transcribe audio (`pyneat.genai.ASRModel`)

ASR turns an audio file into text. This notebook loads the ASR model and transcribes a WAV file with
the **direct** `ASRModel` API.

> Loading the ASR model takes a minute and several GB of memory. Run these cells on the DevKit.

Grounding note: the only shipped ASR **example** in core is server-based
(`/workspace/core/tutorials/021_serve_genai_models/request_audio_transcription.py`, which POSTs to the
server's `/v1/audio/transcriptions`). The **direct** `ASRModel` path below is taken from the public
headers and bindings (`/workspace/core/include/genai/ASRModel.h`, `module.cpp`) and the concept doc
`genai-model.mdx`, which shows exactly this `audio_file` + `language` request shape.

## Two ways to reach ASR - do not confuse them

| Path | Command / API | Covered here |
| --- | --- | --- |
| **CLI smoke test** | `llima run --stt_model_path <elf-dir> <model>` | trap #2, `01-llima-basics/02_...` |
| **Direct Python API** | `pyneat.genai.ASRModel(dir).run(request)` | **this notebook** |
| **HTTP server** | POST to `/v1/audio/transcriptions` on `GenAIServer` | `05-genai-server/` |

**Correctness trap #2 (CLI only):** on the *command line*, ASR needs its ELF directory passed with
`--stt_model_path`. The *Python* `ASRModel` takes the model directory in its constructor and does not
need that flag - the two entry points differ, so do not copy the CLI flag into Python code.

## Memory implications

- `whisper-small-a16w8` is the ASR model used here. Its suffix `a16w8` = **16-bit activations /
  8-bit weights** (8-bit, not 4-bit like the LLM/VLM) - the trade for transcription quality is
  heavier weights per parameter.
- ASR loads an encoder + decoder; memory is dominated by the audio encoder plus the decoder KV cache.
- Transcription cost scales with **audio length**, not a token budget - a long recording takes longer
  and holds more intermediate state.

## Step 1 - Load the ASR model

In [ ]:
import pyneat as neat

MODEL_DIR = "/media/nvme/llima/models/whisper-small-a16w8"

model = neat.genai.ASRModel(MODEL_DIR)
print("model_id     :", model.model_id())
print("accepts_audio:", model.accepts_audio())   # True for an ASR model directory

**Interpretation.** `ASRModel` exposes `model_id()`, `accepts_audio()`, `run()`, and `stream()`
(header `ASRModel.h`). `accepts_audio()` must be `True`. There is no `accepts_text`/`accepts_image`
on `ASRModel` - it is audio-only; use `GenAIModel` if you want a single handle that reports all three
capabilities.

## Step 2 - Transcribe a file

Build a `GenerationRequest`, set `audio_file` to the WAV path and `language`, then `run()`. The
request's `audio_file` / `audio` / `language` fields are the ASR inputs (from `GenAITypes.h`).

In [ ]:
AUDIO_PATH = "/workspace/demo-neat/llima/02-run-llm-vlm/assets/speech.wav"  # your WAV file

request = neat.genai.GenerationRequest()
request.audio_file = AUDIO_PATH      # path to a WAV file
request.language = "en"              # default is "en"

result = model.run(request)
print("transcription:", result.text)
print("tokens:", result.metrics.generated_tokens,
      "| tok/s:", round(result.metrics.tokens_per_second, 2))

**Interpretation.** The result is the same `GenerationResult` type as the LLM/VLM path - `.text`
holds the transcript, `.metrics` holds timing. Two ways to supply audio: `request.audio_file` (a path,
used here) or `request.audio` (an audio `Tensor`, if you already have samples in memory). `language`
defaults to `"en"`; set it to the spoken language for non-English audio.

## Step 3 - Stream a transcription

`ASRModel.stream()` returns a `GenerationStream`, same as the LLM path, so a long recording can print
its transcript progressively.

In [ ]:
stream_request = neat.genai.GenerationRequest()
stream_request.audio_file = AUDIO_PATH
stream_request.language = "en"

stream_handle = model.stream(stream_request)
print("transcription: ", end="", flush=True)
for token in stream_handle:
    print(token.text, end="", flush=True)
    if token.is_final:
        break
print()

**Interpretation.** Streaming is optional for ASR but useful for long audio - you see text as
the decoder produces it instead of waiting for the whole file. Each `TokenSample.text` is the next
transcript fragment.

## The CLI equivalent (trap #2)

The no-code smoke test for the same model:

```bash
# Note --stt_model_path <elf-dir> - required for ASR on the CLI (trap #2):
llima run --stt_model_path /media/nvme/llima/models/whisper-small-a16w8 whisper-small-a16w8
```

The server equivalent (streaming HTTP, from tutorial 021) is:

```bash
python3 .../021_serve_genai_models/request_audio_transcription.py --model asr speech.wav
```

Both are alternatives to the direct `ASRModel` path above; use the direct API in application code.

## Run it as a script

On the DevKit:

```bash
dk /workspace/demo-neat/llima/02-run-llm-vlm/scripts/run_asr.py \
  --model /media/nvme/llima/models/whisper-small-a16w8 \
  --audio /workspace/demo-neat/llima/02-run-llm-vlm/assets/speech.wav
```

Provide your own WAV file (16 kHz mono is a safe choice for whisper). The script errors clearly if the
path is missing.

## Expected output

Running on the DevKit with a real WAV:

- **Load:** `model_id: whisper-small-a16w8`, `accepts_audio: True`.
- **Transcribe:** a `transcription: ...` line containing the spoken words, plus a metrics line.
- **Stream:** the same transcript, printed progressively.

Exact text depends on your audio. If you use the server request script from tutorial 021 instead, it
additionally prints server-side TTFT and average/min/max tokens-per-second.

## References

- `/workspace/core/include/genai/ASRModel.h`, `GenAITypes.h` (`audio_file`, `audio`, `language`)
- `/workspace/core/python/src/module.cpp` (`ASRModel` bindings: `run`, `stream`, `accepts_audio`)
- `/workspace/core/docs/develop-apps/development-workflow/genai-model.mdx` (direct ASR example)
- `/workspace/core/tutorials/021_serve_genai_models/request_audio_transcription.py` (server path)